# Market Risk — full demo (Atoti): VaR, PnL Explain & slice-and-dice

One session, four cubes sharing a **Trades** dimension — the whole story top to bottom:

| Cube | What it shows |
|---|---|
| **Market Risk VaR** | Non-additive VaR, ES, T-1 day-over-day, scenario tail drill, **component** & **incremental** VaR, **backtesting** |
| **PnL Explain** | Attribution by effect / risk class / Greek, cross-tab + zoom — **Market Data uses the same trade delta as the VaR and sensitivities cubes** |
| **VaR by risk** | VaR stacked per desk (Risk class x Tenor) — cells **don't** sum (non-additive); built from the same trades, so desk VaR **reconciles** to the Booking cube |
| **Sensitivities** | Delta/DV01 stacked per desk — **rolled up from the same trade deltas as the VaR cube**; cells **do** sum (additive) + cross-desk netting (Mortgage short = hedge) |

Mirrors ActiveViam's real architecture (separate VaR / PnL / Sensitivity engines glued by trade id). All cubes share **one FRTB risk-class taxonomy** — GIRR; CSR non-Sec (corporate spread); CSR Sec non-CTP (MBS) — so a desk reads the same way everywhere (Mortgage spans GIRR + CSR Sec non-CTP). Every Atoti call verified against the 0.9 docs; data + risk math validated in NumPy.

**Run order:** Part 1 (data) -> Part 2 (session) -> Parts 3-6 (cubes) -> Part 7 (dashboard).


## 0. Install (run once, fresh env)
```bash
conda create -n mr_atoti python=3.11 -y
conda activate mr_atoti
pip install "atoti[jupyterlab]" numpy pandas
```

In [ ]:
import numpy as np
import pandas as pd
from datetime import date

RNG_MAIN = np.random.default_rng(42)   # trades + PnL explain
RNG_RISK = np.random.default_rng(7)    # shared risk sub-vectors -> VaR (trade-level AND risk-slice, reconciled)
RNG_SENS = np.random.default_rng(11)   # sensitivities

N_SCENARIOS    = 250
N_TRADES       = 300
BUSINESS_DATES = pd.bdate_range("2024-06-03", periods=8).date.tolist()
STRESS_DATE    = BUSINESS_DATES[5]     # planted backtest exception
COB            = max(BUSINESS_DATES)   # latest close-of-business (dashboard default)
COB_m1         = sorted(BUSINESS_DATES)[-2]
desks = ["Rates", "Swaps", "Inflation", "Mortgage", "Credit"]

# Part 1 — Synthetic data (five tables)

### 1a. Trades dimension

In [ ]:
ENTITIES = ["US Broker-Dealer", "UK Bank"]
# Books = organizational / mandate units (Mortgage uses real RBC GAT book codes).
# Products = instrument types, on a SEPARATE axis (names don't overlap).
# Book<->Product is many-to-many; for Mortgage it's *constrained* by each book's real mandate (see BOOK_PRODUCTS).
DESK_BOOKS = {"Rates":["Cash","Listed","Repo"], "Swaps":["USD-Swaps","EUR-Swaps","CAD-Swaps"],
              "Inflation":["Real-Rates","Breakevens"], "Mortgage":["CARMA","CSPEC","CTRDA","CIOPO"],
              "Credit":["IG-Cash","HY-Cash","Synthetic"]}
# Swaps / Inflation / Credit: product assigned at desk level.
DESK_PRODUCTS = {"Swaps":["Rates/Interest Rate Swap"],
                 "Inflation":["Fixed Income/Inflation Linker","Rates/Inflation Swap"],
                 "Credit":["Fixed Income/Corporate Bond","Fixed Income/Muni Bond","Credit/CDS"]}
# Rates and Mortgage: product is constrained BY BOOK (real desk mandates), not just by desk.
#   Rates    - Cash = cash Treasuries, Listed = exchange-traded futures, Repo = financing the cash bonds.
#   Mortgage - (RBC GAT) CTRDA trades coupon swaps in pass-throughs/TBAs (NO IO/PO); IO/PO only in CIOPO; CARMA is ARMs.
BOOK_PRODUCTS = {"Cash":["Fixed Income/Treasury Bond"],
                 "Listed":["Rates/Treasury Future"],
                 "Repo":["Fixed Income/Treasury Bond"],          # repo finances the cash treasuries
                 "CARMA":["Securitized/ARM Pass-Through"],
                 "CSPEC":["Securitized/MBS Pass-Through","Securitized/TBA"],
                 "CTRDA":["Securitized/MBS Pass-Through","Securitized/TBA"],
                 "CIOPO":["Securitized/IO","Securitized/PO"]}
CCY = {"US Broker-Dealer":"USD","UK Bank":"GBP"}
DESK_BETA = {"Rates":1.0,"Swaps":0.8,"Inflation":0.4,"Mortgage":-0.6,"Credit":0.7}

rows=[]
for tid in range(N_TRADES):
    desk=desks[tid%len(desks)]; ent=ENTITIES[tid%len(ENTITIES)]
    book=RNG_MAIN.choice(DESK_BOOKS[desk])
    if book in BOOK_PRODUCTS:                                # book-constrained product (Rates + Mortgage)
        ac,prod=RNG_MAIN.choice(BOOK_PRODUCTS[book]).split("/")
    else:                                                    # desk-level product (Swaps/Inflation/Credit)
        ac,prod=DESK_PRODUCTS[desk][tid%len(DESK_PRODUCTS[desk])].split("/")
    ccy = book.split("-")[0] if desk=="Swaps" else CCY[ent]  # swap books are currency-denominated (USD/EUR/CAD)
    rows.append(dict(TradeId=f"T{tid:04d}",Entity=ent,Desk=desk,Book=book,
                     AssetClass=ac,Product=prod,Ccy=ccy))
trades_df=pd.DataFrame(rows); trades_df.head()

### 1b. Shared risk sub-vectors -> trade-level PnL vectors (the reconciled source)

In [ ]:
TENORS=["2Y","5Y","10Y","30Y"]
TENOR_BETA={"2Y":0.3,"5Y":0.6,"10Y":1.0,"30Y":1.4}
# UNIFIED FRTB risk taxonomy, shared by EVERY cube: desk -> [(RiskClass, RiskFactor, [tenors])].
# Risk classes are the Basel FRTB classes: GIRR; CSR non-Sec (corporate spread); CSR Sec non-CTP (MBS/RMBS).
# Inflation is a GIRR risk FACTOR (not its own class); vol is the Vega sensitivity type (Part 6), not a class.
DESK_FRTB={
 "Rates":     [("GIRR","USD Govt Yield",["2Y","5Y","10Y","30Y"])],
 "Swaps":     [("GIRR","USD Swap Curve",["2Y","5Y","10Y"])],
 "Inflation": [("GIRR","USD Inflation",["5Y","10Y"])],
 "Mortgage":  [("GIRR","USD Agency Yield",["5Y","10Y"]),("CSR Sec non-CTP","MBS OAS",["5Y","10Y"])],
 "Credit":    [("CSR non-Sec","IG Spread",["2Y","5Y","10Y"]),("GIRR","USD Govt Yield",["5Y"])],
}
RISK_CLASSES=["GIRR","CSR non-Sec","CSR Sec non-CTP"]

# ONE shared source: every TRADE is decomposed into its desk's (risk class, risk factor, tenor) sub-vectors.
# A trade's PnL vector is the SUM of its sub-vectors -> the VaR cube (Part 3, by Booking) and the
# risk-slice cube (Part 5, by Risk class) are built from the SAME positions, so their desk VaRs reconcile.
# The scenario shock is shared per RISK CLASS (mirrors FRTB intra-class correlation); curves diversify via noise.
sub_rows=[]
for d in BUSINESS_DATES:
    rcc      = {rc: RNG_RISK.standard_normal(N_SCENARIOS) for rc in RISK_CLASSES}  # scenario shock per risk class
    rcc_real = {rc: float(RNG_RISK.standard_normal()) for rc in RISK_CLASSES}      # realized move per risk class
    stress   = -4.5 if d==STRESS_DATE else 0.0
    for t in trades_df.itertuples():
        for rc,rf,tns in DESK_FRTB[t.Desk]:
            for tn in tns:
                scale = RNG_RISK.uniform(3_000,30_000)
                sysf  = DESK_BETA[t.Desk]*TENOR_BETA[tn]                            # desk tilt x tenor sensitivity
                move  = rcc_real[rc]+stress                                         # the risk class's realized factor move
                vec   = np.round(scale*(sysf*rcc[rc] + 0.9*RNG_RISK.standard_normal(N_SCENARIOS)), 2)
                rlz   = scale*(sysf*move + 0.9*float(RNG_RISK.standard_normal()))
                sub_rows.append(dict(AsOfDate=d, TradeId=t.TradeId, Desk=t.Desk, RiskClass=rc, RiskFactor=rf, Tenor=tn,
                                     SubVector=vec, RealizedPnL=round(float(rlz),2),
                                     Delta=round(scale*sysf,2),                     # the position's delta loading -> rolls up to Part 6 AND drives 1c
                                     FactorMove=round(float(move),6)))              # realized factor move -> 1c Market Data PnL = Delta x move
subvec_df=pd.DataFrame(sub_rows)

# Trade-level table = sum each trade's sub-vectors element-wise; realized = sum of the bucket realized.
pnl_df=(subvec_df.groupby(["AsOfDate","TradeId"])
        .agg(PnLVector=("SubVector", lambda s: np.sum(np.stack(s.values),axis=0).tolist()),
             RealizedPnL=("RealizedPnL","sum")).reset_index())
pnl_df["RealizedPnL"]=pnl_df["RealizedPnL"].round(2)
print("sub-vectors", subvec_df.shape, "| trade PnL vectors", pnl_df.shape)

### 1c. PnL Explain — Market Data uses the SAME trade delta as the VaR cube and the sensitivities (one delta everywhere)

In [ ]:
# Market-data attribution reuses the SAME buckets and the SAME Delta as 1b/1e: Sensitivity is the
# position's loading (scale*sysf), and the move is the risk class's realized factor move, so
# PnL = Delta * move equals the SYSTEMATIC part of that bucket's realized P&L. One delta now flows through
# the VaR cube (loading), the sensitivities cube (roll-up), and PnL Explain (attribution).
# The demo P&L model is linear, so only Delta ties; second-order Greeks (Vega/Curvature) are omitted, not faked.
md_cols = subvec_df[["AsOfDate","TradeId","RiskClass","RiskFactor","Tenor","Delta","FactorMove"]].copy()
md_cols.insert(2,"Effect","Market Data"); md_cols["Greek"]="Delta"
md_cols = md_cols.rename(columns={"Delta":"Sensitivity"})
md_cols["MktPrev"]=0.0; md_cols["MktCurr"]=md_cols["FactorMove"].round(6)
md_cols["PnL"]=(md_cols["Sensitivity"]*md_cols["FactorMove"]).round(2)
md_cols = md_cols[["AsOfDate","TradeId","Effect","RiskClass","RiskFactor","Greek","Tenor","Sensitivity","MktPrev","MktCurr","PnL"]]

# Carry/Theta, New Trade, FX, Unexplained: illustrative non-market effects (scalar, not tied to positions).
EFFECTS_SCALAR=["Carry/Theta","New Trade","FX","Unexplained"]
sc_rows=[dict(AsOfDate=d,TradeId=t.TradeId,Effect=eff,RiskClass="N/A",RiskFactor="N/A",Greek="N/A",Tenor="N/A",
             Sensitivity=0.0,MktPrev=0.0,MktCurr=0.0,PnL=round(float(RNG_MAIN.normal(0,3_000)),2))
         for d in BUSINESS_DATES for t in trades_df.itertuples() for eff in EFFECTS_SCALAR]
ex_df=pd.concat([md_cols, pd.DataFrame(sc_rows)], ignore_index=True)
print("pnl explain", ex_df.shape, "| Market Data delta == VaR/sensitivity delta; PnL = Delta x factor move")

### 1d. Risk-slice table — the SAME sub-vectors, tagged by Desk/RiskClass/RiskFactor/Tenor (reconciles to trades)

In [ ]:
# Re-use the shared sub-vectors from 1b: same positions, exposed by (Desk, RiskClass, RiskFactor, Tenor)
# AND TradeId, so this cube reconciles exactly to the trade-level VaR cube.
risk_df = subvec_df[["AsOfDate","TradeId","Desk","RiskClass","RiskFactor","Tenor"]].copy()
risk_df["PnLVector"] = subvec_df["SubVector"].apply(lambda v: v.tolist())
print("risk-slice rows", risk_df.shape)

### 1e. Sensitivities — roll up each trade's Delta (from 1b) to the position level (Book x Risk factor x Tenor)

In [ ]:
# Sensitivities are NO LONGER a separate synthetic population: they are the SAME trade deltas from 1b
# (Delta = scale*sysf, the position's loading on its risk factor), aggregated to position grain.
# A position = a Book's net exposure to a (Risk class, Risk factor, Tenor) bucket = sum of its trades' deltas.
# So Part 6 desk Delta == sum of trade deltas == the loading that drives the VaR cube. Everything ties.
# The demo P&L model is linear, so a faithful roll-up yields DELTA only (no second-order Vega/Curvature to invent).
trade_delta = subvec_df.merge(trades_df[["TradeId","Book"]], on="TradeId")
sens_df = (trade_delta.groupby(["AsOfDate","Desk","Book","RiskClass","RiskFactor","Tenor"], as_index=False)
                      .agg(Sensitivity=("Delta","sum")))
sens_df["Greek"]="Delta"
sens_df["PositionId"] = sens_df["Book"]+" | "+sens_df["RiskFactor"]+" | "+sens_df["Tenor"]
sens_df["Sensitivity"]=sens_df["Sensitivity"].round(2)
sens_df = sens_df[["AsOfDate","PositionId","Desk","Book","RiskClass","RiskFactor","Greek","Tenor","Sensitivity"]]
print("sensitivities (position-level roll-up)", sens_df.shape)

# Part 2 — One session, load all tables, join the shared Trades dimension

In [ ]:
import atoti as tt

from pathlib import Path

# Durable user-content store so saved dashboards survive kernel restart / reboot
CONTENT_DIR = Path(r"C:\Users\bhamb\Yash\NaYa_Fintech_Code\Claude\projects\ActiveViamDemo\content")

# Idempotent start: close a prior in-kernel session so re-running this cell
# doesn't orphan a JVM and lock content.mv.db.
try:
    session.close()
except NameError:
    pass

session = tt.Session.start(
    tt.SessionConfig(user_content_storage=CONTENT_DIR)
)
print("Session started. Content store:", CONTENT_DIR)
print(session.link)
trades_t = session.read_pandas(trades_df, keys=["TradeId"], table_name="Trades")
pnl_t    = session.read_pandas(pnl_df,  keys=["AsOfDate","TradeId"], table_name="PnLVectors")
ex_t     = session.read_pandas(ex_df,   keys=["AsOfDate","TradeId","Effect","RiskClass","RiskFactor","Greek","Tenor"], table_name="PnLExplain")
risk_t   = session.read_pandas(risk_df, keys=["AsOfDate","TradeId","RiskClass","RiskFactor","Tenor"], table_name="RiskVectors")
sens_t   = session.read_pandas(sens_df, keys=["AsOfDate","PositionId"], table_name="Sensitivities")

pnl_t.join(trades_t, pnl_t["TradeId"] == trades_t["TradeId"])
ex_t.join(trades_t,  ex_t["TradeId"]  == trades_t["TradeId"])
session.tables.schema

# Part 3 — Market Risk VaR cube
Non-additive VaR (sum the PnL vectors, *then* quantile) + ES.

In [ ]:
var_cube = session.create_cube(pnl_t, "Market Risk VaR", mode="manual")
hv, lv, mv = var_cube.hierarchies, var_cube.levels, var_cube.measures
hv["Booking"]    = [trades_t["Entity"], trades_t["Desk"], trades_t["Book"], trades_t["TradeId"]]
hv["Product"]    = [trades_t["AssetClass"], trades_t["Product"]]
hv["Currency"]   = [trades_t["Ccy"]]
hv["As-of date"] = [pnl_t["AsOfDate"]]
hv["As-of date"].slicing = True

mv["PnL vector"]   = tt.agg.sum(pnl_t["PnLVector"])
mv["VaR 99%"]      = -tt.array.quantile(mv["PnL vector"], 0.01)
mv["VaR 95%"]      = -tt.array.quantile(mv["PnL vector"], 0.05)
n_tail = int(np.ceil(0.025 * N_SCENARIOS))
mv["ES 97.5%"]     = -tt.array.mean(tt.array.n_lowest(mv["PnL vector"], n_tail))
mv["Realized PnL"] = tt.agg.sum(pnl_t["RealizedPnL"])

# --- backtesting measures (new) ---
mv["Exception"]      = mv["Realized PnL"] < -mv["VaR 99%"]          # boolean, for display
mv["Exception flag"] = tt.where(mv["Exception"], 1, 0)              # int, for summing
mv["Cumulative breaches"] = tt.agg.sum(
    mv["Exception flag"], scope=tt.CumulativeScope(lv["AsOfDate"])
)

for nm in ["VaR 99%","VaR 95%","ES 97.5%","Realized PnL"]: mv[nm].formatter="DOUBLE[#,###]"
list(mv)

### 3a. Headline — diversification (firm VaR < sum of desk VaRs)

In [ ]:
by_desk = var_cube.query(mv["VaR 99%"], mv["ES 97.5%"], levels=[lv["Desk"]], filter=lv["AsOfDate"]==COB)
firm = float(var_cube.query(mv["VaR 99%"], filter=lv["AsOfDate"]==COB)["VaR 99%"].iloc[0])
sumd = float(by_desk["VaR 99%"].astype(float).sum())
print(by_desk, f"\n\nFirm {firm:,.0f} | sum-desks {sumd:,.0f} | diversification {sumd-firm:,.0f} ({100*(1-firm/sumd):.0f}%)")

### 3b. T-1 day-over-day VaR

In [ ]:
mv["VaR 99% (T-1)"]     = tt.shift(mv["VaR 99%"], var_cube.hierarchies["As-of date"], offset=-1)
mv["VaR 99% \u0394 vs T-1"] = mv["VaR 99%"] - mv["VaR 99% (T-1)"]
for nm in ["VaR 99% (T-1)","VaR 99% \u0394 vs T-1"]: mv[nm].formatter="DOUBLE[#,###]"
var_cube.query(mv["VaR 99%"], mv["VaR 99% (T-1)"], mv["VaR 99% \u0394 vs T-1"], levels=[lv["Desk"]], filter=lv["AsOfDate"]==COB)

### 3c. Scenario tail drill — worst-loss scenarios

In [ ]:
if "Scenario" not in var_cube.hierarchies:                       # idempotent: re-running this cell won't add a duplicate
    var_cube.create_parameter_hierarchy_from_members("Scenario", list(range(N_SCENARIOS)))
mv["Scenario PnL"] = mv["PnL vector"][lv["Scenario"]]; mv["Scenario PnL"].formatter="DOUBLE[#,###]"
scen = var_cube.query(mv["Scenario PnL"], levels=[lv["Scenario"]], filter=lv["AsOfDate"]==COB)
scen["Scenario PnL"]=scen["Scenario PnL"].astype(float); scen.sort_values("Scenario PnL").head(8)

### 3d. Component VaR — the *additive* decomposition (sums exactly to firm VaR)

In [ ]:
mv["VaR tail scenario"] = tt.total(tt.array.quantile_index(mv["PnL vector"], 0.01), var_cube.hierarchies["Booking"])
mv["Component VaR"]     = -mv["PnL vector"][mv["VaR tail scenario"]]; mv["Component VaR"].formatter="DOUBLE[#,###]"
comp = var_cube.query(mv["Component VaR"], levels=[lv["Desk"]], filter=lv["AsOfDate"]==COB)
comp["Component VaR"]=comp["Component VaR"].astype(float)
firm_c=float(var_cube.query(mv["Component VaR"], filter=lv["AsOfDate"]==COB)["Component VaR"].iloc[0])
print(comp.round(0), f"\n\nsum of desk components {comp['Component VaR'].sum():,.0f} == firm {firm_c:,.0f}  (negative desks hedge the tail)")

### 3e. Incremental VaR — with-minus-without (non-additive)

In [ ]:
firm_var=float(var_cube.query(mv["VaR 99%"], filter=lv["AsOfDate"]==COB)["VaR 99%"].iloc[0])
inc={}
for dk in desks:
    others=[x for x in desks if x!=dk]
    w=var_cube.query(mv["VaR 99%"], filter=(lv["AsOfDate"]==COB) & lv["Desk"].isin(*others))
    inc[dk]=firm_var-float(w["VaR 99%"].iloc[0])
pd.Series(inc,name="Incremental VaR").round(0).sort_values(ascending=False)

### 3f. Backtesting — exceptions and the Basel traffic light

In [ ]:
bt = var_cube.query(mv["VaR 99%"], mv["Realized PnL"], levels=[lv["AsOfDate"]]).reset_index()
bt["VaR 99%"]=bt["VaR 99%"].astype(float); bt["Realized PnL"]=bt["Realized PnL"].astype(float)
bt=bt.sort_values("AsOfDate")
bt["Exception"]=bt["Realized PnL"] < -bt["VaR 99%"]; bt["Cumulative breaches"]=bt["Exception"].cumsum()
n_exc=int(bt["Exception"].sum()); zone="GREEN" if n_exc<=4 else ("AMBER" if n_exc<=9 else "RED")
print(bt.to_string(index=False)); print(f"\nExceptions {n_exc} -> {zone} zone (Basel zones calibrated for 250 obs)")

# Part 4 — PnL Explain cube
Attribution by effect / risk class / Greek; cross-tab + zoom (drill = zoom).

In [ ]:
pnl_cube = session.create_cube(ex_t, "PnL Explain", mode="manual")
hp, lp, mp = pnl_cube.hierarchies, pnl_cube.levels, pnl_cube.measures
hp["Booking"]     = [trades_t["Entity"], trades_t["Desk"], trades_t["Book"], trades_t["TradeId"]]
hp["Effect"]      = [ex_t["Effect"]]
hp["Risk factor"] = [ex_t["RiskClass"], ex_t["RiskFactor"], ex_t["Tenor"]]
hp["Greek"]       = [ex_t["Greek"]]
hp["As-of date"]  = [ex_t["AsOfDate"]]
hp["As-of date"].slicing = True
mp["PnL Explain"] = tt.agg.sum(ex_t["PnL"]); mp["PnL Explain"].formatter="DOUBLE[#,###]"

ctab = pnl_cube.query(mp["PnL Explain"], levels=[lp["Desk"], lp["Effect"]], filter=lp["AsOfDate"]==COB)
ctab["PnL Explain"]=ctab["PnL Explain"].astype(float); ctab.unstack("Effect").round(0)

### 4a. Zoom (drill deeper, same measure) and drill the market-data effect

In [ ]:
for depth in (["Entity"],["Entity","Desk"],["Entity","Desk","Book"]):
    q=var_cube.query(mv["VaR 99%"], levels=[lv[x] for x in depth], filter=lv["AsOfDate"]==COB)
    print(f"--- zoom: {' > '.join(depth)} ({len(q)} rows) ---"); print(q.head(5),"\n")
print("Market-data effect by risk class x Greek:")
pnl_cube.query(mp["PnL Explain"], levels=[lp["RiskClass"], lp["Greek"]],
               filter=(lp["Effect"]=="Market Data") & (lp["AsOfDate"]==COB))

# Part 5 — VaR slice-and-dice (the stacked tables that *don't* sum)
Card = Desk, rows = Risk class, columns = Tenor, cells = VaR. Cells do NOT sum to the desk total — the gap is diversification.

In [ ]:
rvar_cube = session.create_cube(risk_t, "VaR by risk", mode="manual")
hr, lr, mr = rvar_cube.hierarchies, rvar_cube.levels, rvar_cube.measures
hr["Desk"]=[risk_t["Desk"]]; hr["Risk class"]=[risk_t["RiskClass"]]; hr["Risk factor"]=[risk_t["RiskFactor"]]; hr["Tenor"]=[risk_t["Tenor"]]; hr["As-of date"]=[risk_t["AsOfDate"]]
hr["As-of date"].slicing = True
mr["PnL vector"]=tt.agg.sum(risk_t["PnLVector"]); mr["VaR 99%"]=-tt.array.quantile(mr["PnL vector"],0.01); mr["VaR 99%"].formatter="DOUBLE[#,###]"

def rvar_grid(desk):
    q=rvar_cube.query(mr["VaR 99%"], levels=[lr["RiskClass"], lr["Tenor"]],
                      filter=(lr["AsOfDate"]==COB)&(lr["Desk"]==desk))
    q["VaR 99%"]=q["VaR 99%"].astype(float); return q["VaR 99%"].unstack("Tenor").reindex(columns=TENORS)
for desk in desks:
    g=rvar_grid(desk); dt=float(rvar_cube.query(mr["VaR 99%"], filter=(lr["AsOfDate"]==COB)&(lr["Desk"]==desk))["VaR 99%"].iloc[0])
    print(f"\n===== {desk} desk (VaR) ====="); print(g.round(0).to_string())
    print(f"sum of cells {np.nansum(g.values):,.0f} | desk VaR {dt:,.0f} | diversification {np.nansum(g.values)-dt:,.0f}")

### 5a. Reconciliation — desk VaR is identical via Booking (Part 3) and via Risk class (Part 5)

In [ ]:
recon=[]
for dk in desks:
    v_book=float(var_cube.query(mv["VaR 99%"], filter=(lv["AsOfDate"]==COB)&(lv["Desk"]==dk))["VaR 99%"].iloc[0])
    v_risk=float(rvar_cube.query(mr["VaR 99%"], filter=(lr["AsOfDate"]==COB)&(lr["Desk"]==dk))["VaR 99%"].iloc[0])
    recon.append((dk, v_book, v_risk, round(v_book-v_risk, 2)))
out=pd.DataFrame(recon, columns=["Desk","VaR (Booking cube)","VaR (Risk cube)","diff"])
print(out.round(2).to_string(index=False))
print("\\n-> both cubes aggregate the SAME trade sub-vectors, so desk VaR matches to the cent.")

# Part 6 — Sensitivities slice-and-dice (the stacked tables that *do* sum)
Delta/DV01 by Risk class x Tenor per desk — **rolled up from the same trade deltas that drive the VaR cube**.
Cells sum to margins and totals (additive), and longs/shorts net across desks (Mortgage runs short = the hedge).

In [ ]:
sens_cube = session.create_cube(sens_t, "Sensitivities", mode="manual")
hs, ls, ms = sens_cube.hierarchies, sens_cube.levels, sens_cube.measures
hs["Desk"]=[sens_t["Desk"]]; hs["Book"]=[sens_t["Book"]]; hs["Risk class"]=[sens_t["RiskClass"]]; hs["Risk factor"]=[sens_t["RiskFactor"]]; hs["Greek"]=[sens_t["Greek"]]; hs["Tenor"]=[sens_t["Tenor"]]; hs["As-of date"]=[sens_t["AsOfDate"]]
hs["As-of date"].slicing = True
ms["Sensitivity"]=tt.agg.sum(sens_t["Sensitivity"]); ms["Sensitivity"].formatter="DOUBLE[#,###]"

def delta_grid(desk):
    q=sens_cube.query(ms["Sensitivity"], levels=[ls["RiskClass"], ls["Tenor"]],
                      filter=(ls["AsOfDate"]==COB)&(ls["Desk"]==desk)&(ls["Greek"]=="Delta"))
    q["Sensitivity"]=q["Sensitivity"].astype(float); return q["Sensitivity"].unstack("Tenor").reindex(columns=TENORS)
for desk in desks:
    g=delta_grid(desk); dt=float(sens_cube.query(ms["Sensitivity"], filter=(ls["AsOfDate"]==COB)&(ls["Desk"]==desk)&(ls["Greek"]=="Delta"))["Sensitivity"].iloc[0])
    print(f"\n===== {desk} desk (Delta/DV01) ====="); print(g.round(0).to_string())
    print(f"sum of cells {np.nansum(g.values):,.0f} == desk Delta {dt:,.0f}  (additive)")

### 6a. Cross-desk netting — firm net GIRR Delta by tenor (Mortgage's short offsets the long desks)

In [ ]:
# Restrict to GIRR delta so the netting is dimensionally clean (rate DV01 nets against rate DV01).
base=(ls["AsOfDate"]==COB)&(ls["Greek"]=="Delta")&(ls["RiskClass"]=="GIRR")
bydt=sens_cube.query(ms["Sensitivity"], levels=[ls["Desk"], ls["Tenor"]], filter=base)
bydt["Sensitivity"]=bydt["Sensitivity"].astype(float)
print("GIRR Delta by desk x tenor (Mortgage runs short vs the others long):")
print(bydt["Sensitivity"].unstack("Tenor").reindex(columns=TENORS).round(0).to_string(), "\n")
firm=sens_cube.query(ms["Sensitivity"], levels=[ls["Tenor"]], filter=base); firm["Sensitivity"]=firm["Sensitivity"].astype(float)
print("FIRM NET GIRR Delta by tenor (Mortgage's hedge nets down the total):"); print(firm.reindex(TENORS).round(0).to_string())

# Part 7 — Dashboard and demo script

```python
session.link
```
The web app exposes **all four cubes**. Build pages: (1) VaR by Booking with VaR/ES/Component/Δ-vs-T-1; (2) Backtesting (As-of date rows, VaR vs Realized); (3) PnL Explain (Booking x Effect); (4) VaR slice (Desk pages, Risk class x Tenor); (5) Sensitivities (same, additive). Set the **default filter** on As-of date to the latest COB.

**Demo arc**
1. VaR by desk: firm sits below the sum of desks (diversification).
2. Component VaR sums to firm; Mortgage is a tail hedge. Incremental VaR for the with/without view.
3. ES + scenario drill (the worst day); T-1 delta (move since sign-off); backtest exception + zone.
4. PnL Explain: market data / carry / FX / **unexplained**; drill into risk class -> Greek.
5. **Additive vs non-additive finale**: the Sensitivities deck *sums and nets*; the VaR slice deck *doesn't* — same engine, two behaviours. That's why VaR needs Atoti, not a spreadsheet.


In [ ]:
session.link

In [ ]:
print(session.link)